In [1]:
!pip install python-dotenv langchain langchain-core langchain-community langchain-google-genai chromadb langchain-text-splitters beautifulsoup4 sentence-transformers einops langchainhub langsmith faiss-cpu pydantic rank_bm25 sentence-transformers 
!pip install langchain-google-vertexai python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# In terminal google cloud vertex ai setup
    # !curl https://sdk.cloud.google.com | bash
    # ! gcloud auth application-default login
# !curl "https://aiplatform.googleapis.com/v1/publishers/google/models/gemini-3-flash-preview:streamGenerateContent?key=AQ.Ab8RN6KRw3jXG5kRUuBmsznpQwBfAKRXGyorHhDhg_c7TmxCwQ" -X POST -H "Content-Type: application/json" -d '{"contents":[{"role":"user","parts":[{"text":"Explain how AI works in a few words"}]}]}'


## Declaration

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser
import transformers
import json
import numpy as np
import os
import warnings
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List
from langchain_community.retrievers import BM25Retriever
from langgraph.graph import StateGraph, END
from pydantic import BaseModel
from typing import TypedDict, List
from sentence_transformers import CrossEncoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import time
import vertexai
from langchain_google_vertexai import ChatVertexAI
from sklearn.metrics.pairwise import cosine_similarity



load_dotenv()
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
warnings.filterwarnings("ignore")



warnings.filterwarnings("ignore")


class QueryOutput(BaseModel):
    rewritten_query: str
    expanded_queries: List[str]
    step_back_query: str

class QueryOptimizerOutput(BaseModel):
    rewritten_query: str
    expanded_queries: List[str]
    step_back_query: str

class GraphState(TypedDict):
    query: str
    rewritten_query: str
    expanded_queries: List[str]
    step_back_query: str
    documents: List[Document]
    answer: str
    claims: List[str]                 # NEW
    plan: dict                        # NEW
    iteration: int
    retrieval_feedback: dict
    doc_scores: List[float]



/Users/anideepkalia/Desktop/PolyQuery-AI/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/anideepkalia/Desktop/PolyQuery-AI/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Models

In [ ]:
vertexai.init(project="pleet-d8210", location="global")
llm_compression = ChatVertexAI( model="gemini-3-flash-preview" )
llm = ChatVertexAI( model="gemini-3-flash-preview" )

structured_llm = llm.with_structured_output(QueryOutput)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1",model_kwargs={"trust_remote_code": True})


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13912.99it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
<All keys matched successfully>


In [ ]:
import random

def safe_llm_invoke(llm, prompt, retries=5):
    last_error = None

    for i in range(retries):
        try:
            return llm.invoke(prompt)
        except Exception as e:
            last_error = e
            wait_time = (2 ** i) + random.uniform(0, 1)
            print(f"[Retry {i+1}] Waiting {wait_time:.2f}s →", repr(e))
            time.sleep(wait_time)

    raise last_error

## Query Rewritting & Expansion

* User Query  ->  Query Rewriting  ->  Step-Back Query  ->  Query Expansion


| Step      | Why                                      |
| --------- | ---------------------------------------- |
| Rewrite   | cleans the query and removes ambiguity   |
| Step-back | captures **higher-level concept**        |
| Expansion | generates **multiple search variations** |


In [ ]:
optimizer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert query optimizer. Return ONLY JSON."),
    ("human", """
        User Query:
        {query}

        Perform:
        1. Rewrite query
        2. Generate 3 variations
        3. Generate step-back query

        Return JSON:

        {{
        "rewritten_query": "...",
        "expanded_queries": ["...", "...", "...", "..."],
        "step_back_query": "..."
        }}
    """)
])

def query_intelligence_node(state):

    iteration = state.get("iteration", 0)

    if iteration == 0:
        query = state["query"]
        prompt = optimizer_prompt.invoke({"query": query})
    else:
        feedback = state.get("retrieval_feedback", {})
        snippets = [
            doc.page_content[:200]
            for doc in state.get("documents", [])[:2]
        ]

        refined_query = f"""
            Previous retrieval failed.

            Reason: {feedback.get("reason")}
            Max Score: {feedback.get("max_score")}
            Avg Score: {feedback.get("avg_score")}
            Missing Information: {feedback.get("missing_information")}
            Unsupported Sub-questions: {feedback.get("unsupported_sub_questions")}
            Coverage: {feedback.get("coverage")}

            The following snippets were retrieved but are NOT relevant:
            {chr(10).join(snippets)}

            IMPORTANT:
            - These snippets are incorrect or irrelevant
            - Do NOT base your query on them
            - Use them only to understand what went wrong

            Original Query:
            {state["query"]}

            Previous Rewritten Query:
            {state.get("rewritten_query")}

            Previous Expanded Query:
            {state.get("expanded_queries")}

            Previous Step_back Query:
            {state.get("step_back_query")}

            Your task:
            - Identify why retrieval failed
            - Fix the query without drifting away from user intent
            - Preserve original intent strictly
            - Improve specificity and keywords
        """
        prompt = optimizer_prompt.invoke({"query": refined_query})

    result = safe_llm_invoke(structured_llm, prompt)

    return {
        "rewritten_query": result.rewritten_query,
        "expanded_queries": result.expanded_queries,
        "step_back_query": result.step_back_query
    }



## VECTOR-DB, EMBEDDINGS & INGESTION

In [ ]:
def load_documents_from_txt(file_path):
    documents = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    
    for line in lines:
        line = line.strip()
        if line:  # skip empty lines
            documents.append(Document(page_content=line))
    
    return documents

file_path = "data.txt"
documents = load_documents_from_txt(file_path)


vectorstore = Chroma.from_documents(
    documents=documents,
    collection_name="verirag-chroma",
    embedding=embeddings,
)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 5

## RETRIEVER

In [ ]:
# Retrieve top 5 results from both BM25 and Vector retrievers, then deduplicate results
def hybrid_retrieve_node(state):
    queries = (
        [state["rewritten_query"]] +
        state["expanded_queries"] +
        [state["step_back_query"]]
    )

    all_docs = []
    print("----------------------------")
    for q in queries:
        print("Query: ",q)
        all_docs.extend(bm25_retriever.invoke(q))
        all_docs.extend(vector_retriever.invoke(q))
        print(all_docs)
        
    print("----------------------------")
    # Deduplicate
    unique_docs = list({doc.page_content: doc for doc in all_docs}.values())

    return {"documents": unique_docs}

## Cross Encoder & Re-Ranking

In [ ]:
# Rerank retrieved documents using cross-encoder, normalised scores and filter out low-relevance docs based on a threshold
def rerank_and_filter_node(state):

    docs = state.get("documents", [])
    query = state["query"]

    if not docs:
        return {
            "documents": [],
            "answer": "I don't know",
            "retrieval_feedback": {"reason": "no_docs"}
        }

    pairs = [(query, doc.page_content) for doc in docs]
    scores = cross_encoder.predict(pairs)
    scores = 1 / (1 + np.exp(-scores))

    doc_scores = list(zip(docs, scores))
    doc_scores.sort(key=lambda x: x[1], reverse=True)

    top_docs = doc_scores[:5]
    top_scores = [score for _, score in top_docs]
    max_score = max(top_scores)
    avg_score = sum(top_scores) / len(top_scores)
    print(top_scores, max_score, avg_score)
    print("----------------------------")

    print("rerank: ",[doc for doc, _ in top_docs])
    print("--------------------------------------")

    upper_threshold = 0.7
    lower_threshold = 0.4
    

    if max_score < lower_threshold:
        print("below lower threshold")
        return {
            "documents": [doc for doc, _ in top_docs],
            "answer": "I don't know"
        }

    elif max_score < upper_threshold:
        print("below upper threshold")
        return {
            "documents": [doc for doc, _ in top_docs],
            "doc_scores": top_scores,
            "retrieval_feedback": {
                "reason": "low_relevance",
                "max_score": float(max_score),
                "avg_score": float(avg_score)
            }
        }
    else:
        print("above upper threshold")
        return {
            "documents": [doc for doc, _ in top_docs],
            "doc_scores": top_scores
        }

In [ ]:
def Retry_decision_logic(state):

    iteration = state.get("iteration", 0)
    max_iterations = 2

    if state.get("retrieval_feedback") != None and iteration < max_iterations:
        return "retry"

    # If no docs → retry
    if not state.get("documents") and iteration < max_iterations:
        return "retry"

    return "generate"

def Refine_query_node(state):
    return {
        "iteration": state.get("iteration", 0) + 1
    }

## Compression Node

In [ ]:
def extract_text(response):
    content = response.content

    if isinstance(content, list):
        return " ".join([item.get("text", "") for item in content]).strip()
    
    return content.strip()


def compress_documents_node(state):

    query = state["query"]
    docs = state.get("documents", [])[:3]

    if not docs:
        return {"documents": []}

    # Embed query once
    query_embedding = embeddings.embed_query(query)

    compressed_docs = []

    total_original_tokens = 0
    total_compressed_tokens = 0

    for doc in docs:

        text = doc.page_content

        # Rough token count (interview acceptable approximation)
        total_original_tokens += len(text.split())

        # Step 1: sentence splitting (simple but effective)
        sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 10]

        if not sentences:
            compressed_docs.append(doc)
            continue

        # Step 2: embed sentences
        sentence_embeddings = embeddings.embed_documents(sentences)

        # Step 3: similarity scoring
        scores = cosine_similarity(
            [query_embedding],
            sentence_embeddings
        )[0]

        # Step 4: pick top-k sentences
        top_k = min(3, len(sentences))

        top_indices = np.argsort(scores)[-top_k:]
        top_indices = sorted(top_indices)  # preserve order

        selected_sentences = [sentences[i] for i in top_indices]

        compressed_text = ". ".join(selected_sentences)

        total_compressed_tokens += len(compressed_text.split())

        compressed_docs.append(Document(page_content=compressed_text))

    # 🔥 LOGGING (THIS is what impresses interviewers)
    print("📉 Compression Stats:")
    print(f"Original tokens: {total_original_tokens}")
    print(f"Compressed tokens: {total_compressed_tokens}")

    if total_original_tokens > 0:
        reduction = 1 - (total_compressed_tokens / total_original_tokens)
        print(f"Reduction: {reduction:.2%}")
    
    return {"documents": compressed_docs}





## Answer Planning Node

In [ ]:
def build_empty_plan():
    return {
        "sub_questions": [],
        "required_info": [],
        "evidence_mapping": [],
        "expected_claims": [],
        "supported_claims": [],
        "missing_information": [],
        "risk_flags": ["insufficient_evidence"],
        "answer_mode": "abstain",
        "ordered_evidence": [],
        "coverage": {
            "supported_sub_questions": 0,
            "total_sub_questions": 0,
            "sub_question_coverage": 0.0,
            "supported_claims": 0,
            "total_expected_claims": 0,
            "claim_support_ratio": 0.0
        }
    }


def parse_json_response(response, fallback):
    raw = extract_text(response)

    try:
        return json.loads(raw)
    except Exception:
        start = raw.find("{")
        end = raw.rfind("}")

        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(raw[start:end + 1])
            except Exception:
                pass

    return fallback


def normalize_list(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]

    if isinstance(value, str) and value.strip():
        return [value.strip()]

    return []


def build_sentence_pool(docs):
    sentence_pool = []

    for doc_id, doc in enumerate(docs, start=1):
        sentences = [
            sentence.strip()
            for sentence in doc.page_content.replace("\n", " ").split(".")
            if len(sentence.strip()) > 10
        ]

        for sentence in sentences:
            sentence_pool.append({
                "doc_id": f"DOC_{doc_id}",
                "text": sentence
            })

    return sentence_pool


def rank_sentence_pool(query, sentence_pool, sentence_embeddings, top_k=2):
    if not sentence_pool or len(sentence_embeddings) == 0:
        return []

    query_embedding = embeddings.embed_query(query)
    scores = cosine_similarity([query_embedding], sentence_embeddings)[0]
    ranked_indices = np.argsort(scores)[::-1]

    ranked = []
    for idx in ranked_indices[:top_k]:
        ranked.append({
            "doc_id": sentence_pool[idx]["doc_id"],
            "text": sentence_pool[idx]["text"],
            "score": round(float(scores[idx]), 4)
        })

    return ranked


def answer_planning_node(state):
    query = state["query"]
    docs = state.get("documents", [])
    empty_plan = build_empty_plan()

    if not docs:
        return {
            "plan": empty_plan,
            "retrieval_feedback": {
                "reason": "planner_found_no_documents",
                "missing_information": ["No retrieved evidence available for answer planning."],
                "unsupported_sub_questions": [query]
            }
        }

    context_blocks = [
        f"[DOC_{idx}] {doc.page_content}"
        for idx, doc in enumerate(docs, start=1)
    ]
    context = "\n\n".join(context_blocks)

    prompt = f"""
You are an answer planning engine for a RAG system.

Your job is to create a plan that can be executed before final generation.

Given:
Question: {query}

Context:
{context}

Instructions:
1. Break the question into 2 to 4 answerable sub-questions
2. List the key information required
3. List atomic factual claims that a good answer would make
4. If the context is missing key evidence or looks conflicting, set answer_mode to "abstain"
5. Use ONLY the given context

Return STRICT JSON:

{{
  "sub_questions": ["...", "..."],
  "required_info": ["...", "..."],
  "expected_claims": ["...", "..."],
  "missing_information": ["..."],
  "risk_flags": ["insufficient_evidence"],
  "answer_mode": "grounded_answer"
}}
"""
    response = safe_llm_invoke(llm, prompt)
    plan = parse_json_response(response, build_empty_plan())

    plan["sub_questions"] = normalize_list(plan.get("sub_questions"))
    plan["required_info"] = normalize_list(plan.get("required_info"))
    plan["expected_claims"] = normalize_list(plan.get("expected_claims"))
    plan["missing_information"] = normalize_list(plan.get("missing_information"))
    plan["risk_flags"] = normalize_list(plan.get("risk_flags"))

    if len(plan["sub_questions"]) < 2:
        plan["sub_questions"] = [
            f"What is the direct answer to: {query}?",
            "Which retrieved facts support that answer?"
        ]

    if not plan["required_info"]:
        plan["required_info"] = [f"Evidence needed to answer: {query}"]

    plan["answer_mode"] = "abstain" if plan.get("answer_mode") == "abstain" else "grounded_answer"

    sentence_pool = build_sentence_pool(docs)
    sentence_texts = [item["text"] for item in sentence_pool]
    sentence_embeddings = embeddings.embed_documents(sentence_texts) if sentence_texts else []

    min_support_score = 0.3
    evidence_mapping = []
    ordered_evidence = []
    supported_sub_questions = []

    for sub_question in plan["sub_questions"]:
        ranked = rank_sentence_pool(sub_question, sentence_pool, sentence_embeddings, top_k=2)
        supporting = [item for item in ranked if item["score"] >= min_support_score]

        if supporting:
            supported_sub_questions.append(sub_question)

        evidence_mapping.append({
            "sub_question": sub_question,
            "supporting_sentences": [item["text"] for item in supporting],
            "source_docs": [item["doc_id"] for item in supporting],
            "support_score": supporting[0]["score"] if supporting else 0.0
        })

        for item in supporting:
            if item["text"] not in ordered_evidence:
                ordered_evidence.append(item["text"])

    supported_claims = []
    for claim in plan["expected_claims"]:
        ranked = rank_sentence_pool(claim, sentence_pool, sentence_embeddings, top_k=1)

        if ranked and ranked[0]["score"] >= min_support_score:
            supported_claims.append(claim)

            if ranked[0]["text"] not in ordered_evidence:
                ordered_evidence.append(ranked[0]["text"])

    if not supported_claims and ordered_evidence:
        supported_claims = ordered_evidence[:min(3, len(ordered_evidence))]

    sub_question_coverage = round(
        len(supported_sub_questions) / max(1, len(plan["sub_questions"])),
        2
    )
    claim_support_ratio = round(
        len(supported_claims) / max(1, len(plan["expected_claims"])),
        2
    ) if plan["expected_claims"] else round(float(bool(supported_claims)), 2)

    plan["evidence_mapping"] = evidence_mapping
    plan["supported_claims"] = supported_claims
    plan["ordered_evidence"] = ordered_evidence
    plan["coverage"] = {
        "supported_sub_questions": len(supported_sub_questions),
        "total_sub_questions": len(plan["sub_questions"]),
        "sub_question_coverage": sub_question_coverage,
        "supported_claims": len(supported_claims),
        "total_expected_claims": len(plan["expected_claims"]),
        "claim_support_ratio": claim_support_ratio
    }

    risk_flags = set(plan["risk_flags"])

    if sub_question_coverage < 1.0 or not ordered_evidence:
        risk_flags.add("insufficient_evidence")

    if plan["expected_claims"] and claim_support_ratio < 1.0:
        risk_flags.add("unsupported_claims")

    plan["risk_flags"] = sorted(risk_flags)

    if (
        "conflicting_evidence" in plan["risk_flags"]
        or plan["answer_mode"] == "abstain"
        or sub_question_coverage < 0.5
        or not supported_claims
        or not ordered_evidence
    ):
        plan["answer_mode"] = "abstain"
    else:
        plan["answer_mode"] = "grounded_answer"

    print("🧠 Answer Plan Summary:", {
        "answer_mode": plan["answer_mode"],
        "coverage": plan["coverage"],
        "risk_flags": plan["risk_flags"],
        "supported_claims": plan["supported_claims"]
    })

    if plan["answer_mode"] == "abstain" and "insufficient_evidence" in plan["risk_flags"]:
        unsupported = [
            item["sub_question"]
            for item in evidence_mapping
            if not item["supporting_sentences"]
        ]

        return {
            "plan": plan,
            "retrieval_feedback": {
                "reason": "planner_detected_missing_evidence",
                "missing_information": plan["missing_information"],
                "unsupported_sub_questions": unsupported or plan["sub_questions"],
                "coverage": plan["coverage"]
            }
        }

    return {"plan": plan}


def planning_decision_logic(state):
    iteration = state.get("iteration", 0)
    max_iterations = 2
    plan = state.get("plan", {})
    risk_flags = plan.get("risk_flags", [])

    if (
        plan.get("answer_mode") == "abstain"
        and "insufficient_evidence" in risk_flags
        and iteration < max_iterations
    ):
        return "retry"

    return "generate"



## GENERATION

In [ ]:
def generate_answer_node(state):

    if state.get("answer") == "I don't know":
        return {"answer": "I don't know", "claims": []}

    docs = state.get("documents", [])
    plan = state.get("plan", build_empty_plan())

    if not docs:
        return {"answer": "I don't know", "claims": []}

    if not plan or plan.get("answer_mode") != "grounded_answer":
        print("Generator abstained due to plan:", plan.get("risk_flags", []))
        return {"answer": "I don't know", "claims": []}

    ordered_evidence = plan.get("ordered_evidence", [])
    supported_claims = plan.get("supported_claims", [])

    if not ordered_evidence or not supported_claims:
        return {"answer": "I don't know", "claims": []}

    prompt = f"""
You are a strict grounded answer generator.

Question:
{state["query"]}

Sub-questions:
{json.dumps(plan.get("sub_questions", []), indent=2)}

Evidence mapping:
{json.dumps(plan.get("evidence_mapping", []), indent=2)}

Allowed evidence sentences:
{json.dumps(ordered_evidence, indent=2)}

Supported claims you MAY use:
{json.dumps(supported_claims, indent=2)}

Instructions:
- Use ONLY the allowed evidence sentences
- Answer the question directly and concisely
- Keep claims atomic and factual
- Claims must stay within the supported_claims list
- If the evidence is not enough, return \"I don't know\"

Return STRICT JSON:

{{
  "answer": "...",
  "claims": ["...", "..."]
}}
"""

    response = safe_llm_invoke(llm, prompt)
    result = parse_json_response(response, {"answer": "I don't know", "claims": []})

    result["claims"] = normalize_list(result.get("claims"))
    allowed_claims = set(supported_claims)
    filtered_claims = [claim for claim in result["claims"] if claim in allowed_claims]

    if not filtered_claims:
        filtered_claims = supported_claims

    if not filtered_claims:
        return {"answer": "I don't know", "claims": []}

    return {
        "answer": " ".join(filtered_claims),
        "claims": filtered_claims
    }



## Graph Compliation

In [ ]:
builder = StateGraph(GraphState)

builder.add_node("query_intelligence", query_intelligence_node)
builder.add_node("hybrid_retrieve", hybrid_retrieve_node)
builder.add_node("rerank", rerank_and_filter_node)
builder.add_node("Refine_query", Refine_query_node)
builder.add_node("Generate", generate_answer_node)
builder.add_node("Plan", answer_planning_node)
builder.add_node("Compress", compress_documents_node)

builder.set_entry_point("query_intelligence")

builder.add_edge("query_intelligence", "hybrid_retrieve")
builder.add_edge("hybrid_retrieve", "rerank")
builder.add_edge("Compress", "Plan")

builder.add_conditional_edges(
    "rerank",
    Retry_decision_logic,
    {
        "retry": "Refine_query",
        "generate": "Compress"
    }
)

builder.add_conditional_edges(
    "Plan",
    planning_decision_logic,
    {
        "retry": "Refine_query",
        "generate": "Generate"
    }
)

builder.add_edge("Refine_query", "query_intelligence")
builder.add_edge("Generate", END)

graph = builder.compile()

from IPython.display import Image, display # type: ignore
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))



## Inferencing RAG

In [ ]:
result = graph.invoke({
    "query": "How do we measure faithfulness in computer terminal?",
    # "query": "How does RAG ensure grounded generation despite irrelevant retrieved documents?",
    # "query": "How do we measure faithfulness in RAG LLM?",
    # "query": "how tall is the Eiffel Tower ?",
    # "query": "What is a RAG system?",
    # "query": "What is my favorite color?",
    # "query": "What is my name??",
    "iteration": 0
})

raw = result.get("answer", "")

if isinstance(raw, list) and len(raw) > 0:
    print(raw[0].get("text", ""))
else:
    print(raw)

print("Claims:", result.get("claims", []))
print("Planner Mode:", result.get("plan", {}).get("answer_mode"))
print("Planner Coverage:", result.get("plan", {}).get("coverage"))

